## **English To Tamil Translator**
*MODEL_NAME = "facebook/nllb-200-distilled-600M"*

In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 104.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:


In [ ]:

!pip install -q transformers sentencepiece gradio

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings("ignore")



In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)

tokenizer.src_lang = "eng_Latn"

print("Model ready!")
print("Device:", device)

In [ ]:

def translate(text):
    if not text.strip():
        return "Please enter some English text."

    tokenizer.src_lang = "eng_Latn"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    tamil_token_id = tokenizer.convert_tokens_to_ids("tam_Taml")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=tamil_token_id,
            max_new_tokens=256,
            num_beams=4
        )

    result = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return result

print("Translation function ready!")


In [ ]:

english_text = "I am learning artificial intelligence."

tamil_text = translate(english_text)

print("English:", english_text)
print("Tamil:", tamil_text)


In [7]:
import gradio as gr

app = gr.Interface(
    fn=translate,
    inputs=gr.Textbox(
        label="Enter English text",
        placeholder="Type a sentence in English...",
        lines=5
    ),
    outputs=gr.Textbox(
        label="Tamil Translation",
        lines=5
    ),
    title="English to Tamil AI Translator",
    description="Translate English text into Tamil using a pretrained Hugging Face model."
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e25c9873b1dd07a8dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
